# 07 — Evaluation & analysis (EM / F1)

Reads RAG outputs from `RESULTS_DIR`.

In [ ]:
import pandas as pd

rag_results_df_bm25 = pd.read_parquet(answers_path("bm25"))
print("Loaded:", answers_path("bm25"))

In [ ]:
import pandas as pd

rag_results_df_dense = pd.read_parquet(answers_path("dense"))
print("Loaded:", answers_path("dense"))



In [ ]:
import pandas as pd
import pyarrow.parquet as pq

RAG_OUTPUT_PATH_HYBRID = answers_path("hybrid")

# Columns needed for evaluation
good_cols = [
    "question_id",
    "question",
    "generated_answer",
    "gold_answer_text",
]

pf = pq.ParquetFile(RAG_OUTPUT_PATH_HYBRID)

data = {name: [] for name in good_cols}

for rg_index in range(pf.num_row_groups):
    rg_table = pf.read_row_group(rg_index, columns=good_cols)
    rg_table = rg_table.combine_chunks()
    for name in good_cols:
        col = rg_table[name]
        data[name].extend(col.to_pylist())

rag_results_df_hybrid = pd.DataFrame(data)
print("Loaded:", RAG_OUTPUT_PATH_HYBRID)

In [ ]:
import re
import string
import numpy as np

def normalize_text(s: str) -> str:
    """
    Lowercase, remove punctuation, articles, and extra whitespace.
    This matches common EM/F1 implementations used in QA benchmarks.
    """
    if s is None:
        return ""

    # Lowercase
    s = s.lower()

    # Remove punctuation
    exclude = set(string.punctuation)
    s = "".join(ch for ch in s if ch not in exclude)

    # Remove articles
    s = re.sub(r"\b(a|an|the)\b", " ", s)

    # Normalize whitespace
    s = " ".join(s.split())

    return s


In [ ]:
def exact_match_score(pred: str, gold: str) -> float:
    """
    1.0 if normalized prediction == normalized gold, else 0.0.
    """
    return float(normalize_text(pred) == normalize_text(gold))


def f1_score_qa(pred: str, gold: str) -> float:
    """
    Token-level F1 between prediction and a single gold answer.
    """
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = {}
    for t in pred_tokens:
        if t in gold_tokens:
            common[t] = common.get(t, 0) + 1

    num_common = 0
    for t in common:
        num_common += min(pred_tokens.count(t), gold_tokens.count(t))

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall    = num_common / len(gold_tokens)
    f1        = 2 * precision * recall / (precision + recall)
    return f1


In [ ]:
def get_gold_answers(row) -> list[str]:
    """
    Turn gold_answer_text into a list of gold answer strings.
    Supports:
      - already a list
      - a string with answers separated by " , "
      - missing/NaN → empty list
    """
    val = row.get("gold_answer_text", None)

    # If already a list
    if isinstance(val, list):
        return [str(a).strip() for a in val if str(a).strip()]

    # If NaN / missing
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []

    # Otherwise treat as a string with " , " delimiter
    s = str(val)
    parts = [p.strip() for p in s.split(" , ") if p.strip()]
    return parts


def best_em_f1_for_row(pred: str, golds: list[str]) -> tuple[float, float]:
    """
    For one prediction and a list of gold answers, return:
      (best_exact_match, best_f1)
    """
    if not golds:
        return np.nan, np.nan  # skip in aggregate

    em_scores = [exact_match_score(pred, g) for g in golds]
    f1_scores = [f1_score_qa(pred, g) for g in golds]

    return max(em_scores), max(f1_scores)


def clean_pred(pred: str) -> str:
    if pred is None:
        return ""
    pred = pred.strip()

    pred = re.sub(r"^(the answer is|answer is|answer:)\s*", "", pred, flags=re.I)
    pred = pred.strip(" \"'.,;:")

    return pred


def add_em_f1_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute EM and F1 for each row and add them as 'em' and 'f1' columns.
    Modifies df in-place and also returns it.
    """
    em_list = []
    f1_list = []

    for _, row in df.iterrows():
        pred = clean_pred(row["generated_answer"])
        gold_answers = get_gold_answers(row)

        em, f1 = best_em_f1_for_row(pred, gold_answers)
        em_list.append(em)
        f1_list.append(f1)

    df["em"] = em_list
    df["f1"] = f1_list
    return df


In [ ]:
# Drop rows with NaN EM/F1 (questions that had no gold answers)
rag_results_df_bm25 = add_em_f1_metrics(rag_results_df_bm25)

# keep only rows where EM/F1 are defined
valid_df = rag_results_df_bm25.dropna(subset=["em", "f1"])

avg_em = valid_df["em"].mean()
avg_f1 = valid_df["f1"].mean()

print("BM25 Results are as follows:")
print(f"Number of evaluated questions: {len(valid_df)}")
print(f"Exact Match (EM):                 {avg_em:.4f}")
print(f"F1:                               {avg_f1:.4f}")





In [ ]:
# Drop rows with NaN EM/F1 (questions that had no gold answers)
rag_results_df_dense = add_em_f1_metrics(rag_results_df_dense)

# keep only rows where EM/F1 are defined
valid_df = rag_results_df_dense.dropna(subset=["em", "f1"])

avg_em = valid_df["em"].mean()
avg_f1 = valid_df["f1"].mean()

print(f"Number of evaluated questions: {len(valid_df)}")
print(f"Exact Match (EM):                 {avg_em:.4f}")
print(f"F1:                               {avg_f1:.4f}")



In [ ]:
# Drop rows with NaN EM/F1 (questions that had no gold answers)
rag_results_df_hybrid = add_em_f1_metrics(rag_results_df_hybrid)

# keep only rows where EM/F1 are defined
valid_df = rag_results_df_hybrid.dropna(subset=["em", "f1"])


avg_em = valid_df["em"].mean()
avg_f1 = valid_df["f1"].mean()

print(f"Number of evaluated questions: {len(valid_df)}")
print(f"Exact Match (EM):                 {avg_em:.4f}")
print(f"F1:                               {avg_f1:.4f}")



In [ ]:
for name, d in [("BM25", rag_results_df_bm25), ("Dense", rag_results_df_dense), ("Hybrid", rag_results_df_hybrid)]:
    v = d.dropna(subset=["em", "f1"])
    append_result({
        "retriever": name,
        "generator": GEN_TAG,
        "chunk_size": CHUNK_SIZE,
        "overlap_ratio": OVERLAP_RATIO,
        "n_questions": len(v),
        "EM": v["em"].mean(),
        "F1": v["f1"].mean(),
    }, GENERATION_SUMMARY_CSV)

print("Saved generation results to:", GENERATION_SUMMARY_CSV)